In [3]:
#Importar Biblioteca

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, upper, trim, coalesce,
    datediff, expr, row_number, when
)
from pyspark.sql.window import Window

StatementMeta(, 4bbc25f9-6440-4fc3-be3e-f226fd78a935, 5, Finished, Available, Finished, False)

In [4]:
#Ler LakeHouse_Bronze tabela Fato

df_bronze = spark.read.table("LakeHouse_Bronze.dbo.fato_ocorrencias")

StatementMeta(, 4bbc25f9-6440-4fc3-be3e-f226fd78a935, 6, Finished, Available, Finished, False)

In [5]:
#Remover duplicatas

window_spec = Window.partitionBy("nr_ocorrencia") \
                    .orderBy(col("data_atualizacao").desc())

df_dedup = df_bronze.withColumn(
    "rn",
    row_number().over(window_spec)
).filter(col("rn") == 1).drop("rn")

StatementMeta(, 4bbc25f9-6440-4fc3-be3e-f226fd78a935, 7, Finished, Available, Finished, False)

In [6]:
#Transformações e Tratamentos

df_silver = df_dedup.select(
    col("nr_ocorrencia").cast("long"),
    col("data_chamado").cast("timestamp"),
    col("data_resposta").cast("timestamp"),

    # tempo_resposta em segundos
    coalesce(
        col("tempo_resposta_s_").cast("int"),
        (col("data_resposta").cast("long") - col("data_chamado").cast("long"))
    ).alias("tempo_resposta_seg"),

    col("id_usuario").cast("int"),
    col("id_problema").cast("int"),
    col("id_suporte").cast("int"),

    upper(trim(col("status"))).alias("status"),

    col("data_atualizacao").cast("timestamp")
)

StatementMeta(, 4bbc25f9-6440-4fc3-be3e-f226fd78a935, 8, Finished, Available, Finished, False)

In [7]:
#Criar Colunas Derivadas

df_silver = df_silver.withColumn(
    "tempo_resposta_min",
    col("tempo_resposta_seg") / 60.0
).withColumn(
    "sla_atendido",
    when(col("tempo_resposta_seg") <= 3600, 1).otherwise(0)
)

StatementMeta(, 4bbc25f9-6440-4fc3-be3e-f226fd78a935, 9, Finished, Available, Finished, False)

In [8]:
#Validar Dados Críticos

df_silver = df_silver.filter(
    col("data_chamado").isNotNull()
)

StatementMeta(, 4bbc25f9-6440-4fc3-be3e-f226fd78a935, 10, Finished, Available, Finished, False)

In [9]:
#Apenas na primeira carga 
#df_silver.write \
#    .format("delta") \
#    .mode("overwrite") \
#    .saveAsTable("LakeHouse_Silver.dbo.fato_ocorrencias")

StatementMeta(, 4bbc25f9-6440-4fc3-be3e-f226fd78a935, 11, Finished, Available, Finished, False)

In [10]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "LakeHouse_Silver.dbo.fato_ocorrencias"
)

delta_table.alias("target").merge(
    df_silver.alias("source"),
    "target.nr_ocorrencia = source.nr_ocorrencia"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

StatementMeta(, 4bbc25f9-6440-4fc3-be3e-f226fd78a935, 12, Finished, Available, Finished, False)

In [11]:
#df = spark.sql("SELECT * FROM LakeHouse_Silver.dbo.fato_ocorrencias LIMIT 1000") display(df)

StatementMeta(, 4bbc25f9-6440-4fc3-be3e-f226fd78a935, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9c27baf2-f7b0-4988-b18b-5a3dc1e03b4f)